# 1. Spark processing on EMR

## Setting up PySpark on an EMR Cluster

Before executing this Jupyter notebook, I launched a Spark-enabled AWS EMR cluster using the following command in Terminal.

```bash
python launch_spark_cluster.py --s3_bucket ***bucketnamehere** --primary_count 1 --core_count 2 --instance_type m5.xlarge
```

ex.
```bash
python launch_spark_cluster.py --s3_bucket nana-survey-bucket-2026 --primary_count 1 --core_count 2 --instance_type m5.xlarge
```


After the EMR cluster was launched, I configured SSH port forwarding to access JupyterHub running on the EMR primary node.

```bash
ssh -i ***.pem" -NL 9443:localhost:9443 hadoop@ec****.compute-1.amazonaws.com
```

ex.
```bash
ssh -i "C:\Users\evano\OneDrive\ドキュメント\GitHub\a4-NanaTakeshiba\labsuser.pem" -NL 9443:localhost:9443 hadoop@ec2-54-210-38-87.compute-1.amazonaws.com
```


I then navigated to `https://localhost:9443` in a web browser and executed this notebook through the remote JupyterHub environment connected to the Spark cluster.

In [1]:
%%configure -f
{
    "conf": {
        "spark.pyspark.python": "python3",
        "spark.pyspark.virtualenv.enabled": "true",
        "spark.pyspark.virtualenv.type":"native",
        "spark.pyspark.virtualenv.bin.path":"/usr/bin/virtualenv"
    }
}

In [2]:
spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
1,application_1780022620047_0003,pyspark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
# imports

import time

from pyspark.sql.functions import (
    col,
    trim,
    lower,
    regexp_replace,
    explode,
    array_remove,
    size
)

from pyspark.ml.feature import (
    Tokenizer,
    StopWordsRemover
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Loading JSON files

In [4]:
posts = spark.read.json(
    "s3://luchen-lab/raw/reddit/posts/source=archive/subreddit=mentalhealth/"
)

comments = spark.read.json(
    "s3://luchen-lab/raw/reddit/comments/source=archive/subreddit=mentalhealth/"
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Checking the style of data

In [5]:
print("Posts count:")
print(posts.count())

print("Comments count:")
print(comments.count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Posts count:
603891
Comments count:
1818218

In [6]:
print("Posts schema:")
posts.printSchema()

print("Comments schema:")
comments.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Posts schema:
root
 |-- created_date: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- ingested_at: string (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- post_id: string (nullable = true)
 |-- score: long (nullable = true)
 |-- selftext: string (nullable = true)
 |-- task_id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

Comments schema:
root
 |-- body: string (nullable = true)
 |-- comment_id: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- ingested_at: string (nullable = true)
 |-- post_id: string (nullable = true)
 |-- score: long (nullable = true)
 |-- task_id: long (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

In [7]:
posts.show(5, truncate=True)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+-----------+--------------------+------------+-------+-----+--------------------+-------+--------------------+----+-----+
|created_date|created_utc|         ingested_at|num_comments|post_id|score|            selftext|task_id|               title|year|month|
+------------+-----------+--------------------+------------+-------+-----+--------------------+-------+--------------------+----+-----+
|  2024-01-01| 1704067204|2026-05-29T00:42:32Z|           0|18vkgs2|    1|I need to free my...|    120|      I need to talk|2024|    1|
|  2024-01-01| 1704067342|2026-05-29T00:42:32Z|          26|18vkikn|    1| You matter. You ...|    120|If you feel alone...|2024|    1|
|  2024-01-01| 1704067712|2026-05-29T00:42:32Z|           0|18vkmic|    1|
Hey coming to th...|    120|Mental health / a...|2024|    1|
|  2024-01-01| 1704068116|2026-05-29T00:42:32Z|           0|18vkqy5|    1|I have never had ...|    120|I keep falling as...|2024|    1|
|  2024-01-01| 1704068308|2026-05-29T00:42:32Z| 

In [8]:
comments.show(5, truncate=True)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+----------+------------+-----------+--------------------+-------+-----+-------+----+-----+
|                body|comment_id|created_date|created_utc|         ingested_at|post_id|score|task_id|year|month|
+--------------------+----------+------------+-----------+--------------------+-------+-----+-------+----+-----+
|Thank you for for...|   hmqawfp|  2021-12-01| 1638316891|2026-05-28T20:32:22Z| r6178v|    1|     71|2021|   12|
|Great article htt...|   hmqbset|  2021-12-01| 1638317288|2026-05-28T20:32:22Z| r60f08|    1|     71|2021|   12|
|I’m really happy ...|   hmqc3q0|  2021-12-01| 1638317432|2026-05-28T20:32:22Z| r5x6iu|    3|     71|2021|   12|
|Thank you and lit...|   hmqcov7|  2021-12-01| 1638317700|2026-05-28T20:32:22Z| r5x6iu|    7|     71|2021|   12|
|It really does, t...|   hmqcq41|  2021-12-01| 1638317715|2026-05-28T20:32:22Z| r5x6iu|    5|     71|2021|   12|
+--------------------+----------+------------+-----------+--------------------+-------+-----+---

# 2 Data Cleaning

In [9]:
posts_clean = posts.filter(
    (col("selftext").isNotNull()) &
    (col("selftext") != "[deleted]") &
    (col("selftext") != "[removed]") &
    (trim(col("selftext")) != "")
)

comments_clean = comments.filter(
    (col("body").isNotNull()) &
    (col("body") != "[deleted]") &
    (col("body") != "[removed]") &
    (trim(col("body")) != "")
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
posts_clean.cache()
comments_clean.cache()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[body: string, comment_id: string, created_date: string, created_utc: bigint, ingested_at: string, post_id: string, score: bigint, task_id: bigint, year: int, month: int]

In [11]:
print("Original posts:", posts.count())
print("Clean posts:", posts_clean.count())

print("Original comments:", comments.count())
print("Clean comments:", comments_clean.count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Original posts: 603891
Clean posts: 399418
Original comments: 1818218
Clean comments: 1818218

# 3. Feature engineering

In [12]:
tokenizer = Tokenizer(
    inputCol="clean_text",
    outputCol="tokens"
)

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
# Posts NLP
posts_nlp = posts_clean.withColumn(
    "clean_text",
    lower(col("selftext"))
)

posts_nlp = posts_nlp.withColumn(
    "clean_text",
    regexp_replace("clean_text", "[^a-zA-Z\\s]", "")
)

posts_nlp = tokenizer.transform(posts_nlp)

posts_nlp = remover.transform(posts_nlp)

posts_nlp = posts_nlp.withColumn(
    "filtered_tokens",
    array_remove("filtered_tokens", "")
)

posts_nlp = posts_nlp.withColumn(
    "num_tokens",
    size("filtered_tokens")
)

posts_nlp.cache()

posts_nlp.select(
    "clean_text",
    "tokens",
    "filtered_tokens"
).show(5, truncate=50)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+
|                                        clean_text|                                            tokens|                                   filtered_tokens|
+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+
|i need to free myself from this burden that eat...|[i, need, to, free, myself, from, this, burden,...|[need, free, burden, eats, daily, deeply, traum...|
| you matter you are enough and im proud of all ...|[, you, matter, you, are, enough, and, im, prou...|[matter, enough, im, proud, efforts, probably, ...|
|
hey coming to the wider community for support ...|[, hey, coming, to, the, wider, community, for,...|[hey, coming, wider, community, support, sugges...|
|i have never had problems with my sleep and if ...|[i, have, never, h

In [14]:
# Comments NLP
comments_nlp = comments_clean.withColumn(
    "clean_text",
    lower(col("body"))
)

comments_nlp = comments_nlp.withColumn(
    "clean_text",
    regexp_replace("clean_text", "[^a-zA-Z\\s]", "")
)

comments_nlp = tokenizer.transform(comments_nlp)

comments_nlp = remover.transform(comments_nlp)

comments_nlp = comments_nlp.withColumn(
    "filtered_tokens",
    array_remove("filtered_tokens", "")
)

comments_nlp = comments_nlp.withColumn(
    "num_tokens",
    size("filtered_tokens")
)

comments_nlp.cache()

comments_nlp.select(
    "clean_text",
    "tokens",
    "filtered_tokens"
).show(5, truncate=50)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+
|                                        clean_text|                                            tokens|                                   filtered_tokens|
+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+
|thank you for for sharing a reminder if you are...|[thank, you, for, for, sharing, a, reminder, if...|[thank, sharing, reminder, seeking, resources, ...|
|great article httpswwwhealthharvardedumindandmo...|[great, article, httpswwwhealthharvardedumindan...|[great, article, httpswwwhealthharvardedumindan...|
|im really happy for you i really hope that weig...|[im, really, happy, for, you, i, really, hope, ...|[im, really, happy, really, hope, weight, lifte...|
|thank you and literally that thought is what ma...|[thank, you, and, 

# 4. Word Frequency Analysis

In [15]:
posts_words = posts_nlp.select(
    explode("filtered_tokens").alias("word")
)

posts_words.groupBy("word").count().orderBy(
    col("count").desc()
).show(20)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+------+
|     word| count|
+---------+------+
|       im|857540|
|     like|622341|
|     dont|524408|
|     feel|489092|
|     know|372302|
|      ive|329178|
|      get|312728|
|     want|284541|
|     time|269015|
|     even|266208|
|   really|265307|
|   people|240547|
|     life|237829|
|     cant|230481|
|   things|196497|
|      one|193423|
|    think|190698|
|     help|181246|
|       go|167251|
|something|162223|
+---------+------+
only showing top 20 rows

In [16]:
comments_words = comments_nlp.select(
    explode("filtered_tokens").alias("word")
)

comments_words.groupBy("word").count().orderBy(
    col("count").desc()
).show(20)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+-------+
|     word|  count|
+---------+-------+
|   please|2004055|
|    thank| 658890|
|     feel| 658400|
|     post| 637386|
|     help| 633198|
|     like| 563278|
|     list| 549251|
|  seeking| 547963|
|community| 546566|
|    local| 545170|
|    click| 541036|
|       im| 517624|
|  someone| 498154|
|     dont| 494156|
|     find| 449731|
|     make| 448922|
|   people| 401623|
|  therapy| 390660|
|      get| 386367|
|     well| 385901|
+---------+-------+
only showing top 20 rows

# 5. Yearly Aggregation

In [17]:
yearly_posts = posts_nlp.groupBy(
    "year"
).count().orderBy(
    "year"
)

yearly_posts.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+------+
|year| count|
+----+------+
|2019| 37868|
|2020| 53039|
|2021| 61627|
|2022| 71196|
|2023| 69349|
|2024|106339|
+----+------+

In [18]:
yearly_comments = comments_nlp.groupBy(
    "year"
).count().orderBy(
    "year"
)

yearly_comments.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+------+
|year| count|
+----+------+
|2019|172849|
|2020|287625|
|2021|387720|
|2022|378014|
|2023|256953|
|2024|335057|
+----+------+

# 6. Average Token Counts

In [19]:
posts_nlp.groupBy(
    "year"
).avg("num_tokens").orderBy(
    "year"
).show()

comments_nlp.groupBy(
    "year"
).avg("num_tokens").orderBy(
    "year"
).show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+------------------+
|year|   avg(num_tokens)|
+----+------------------+
|2019| 106.5616879687335|
|2020|102.48170214370558|
|2021| 99.69983935612638|
|2022| 97.13673520984325|
|2023|102.16781784885146|
|2024|  98.1776676478056|
+----+------------------+

+----+------------------+
|year|   avg(num_tokens)|
+----+------------------+
|2019|28.720652129893722|
|2020|37.840983920034766|
|2021| 43.53837563189931|
|2022|  41.6079589644828|
|2023|32.945846127501916|
|2024|33.552016522561836|
+----+------------------+

# 7. Save Processed Data as Parquet

In [20]:
posts_nlp.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet(
        "s3://nana-survey-bucket-2026/reddit/posts_nlp/"
    )

comments_nlp.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet(
        "s3://nana-survey-bucket-2026/reddit/comments_nlp/"
    )

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# 8. Scalability Benchmark

In [21]:
# Fixed random seed for reproducibility
SEED = 42

fractions = [0.1, 0.5, 1.0]

for frac in fractions:

    print(f"\n Benchmark: {int(frac*100)}% Data ")

    start = time.time()

    # Sample Data

    sample_posts = posts.sample(
        withReplacement=False,
        fraction=frac,
        seed=SEED
    )

    # Cleaning

    sample_clean = sample_posts.filter(
        (col("selftext").isNotNull()) &
        (col("selftext") != "[deleted]") &
        (col("selftext") != "[removed]") &
        (trim(col("selftext")) != "")
    )

    # NLP preprocessing

    sample_nlp = sample_clean.withColumn(
        "clean_text",
        lower(col("selftext"))
    )

    sample_nlp = sample_nlp.withColumn(
        "clean_text",
        regexp_replace(
            "clean_text",
            "[^a-zA-Z\\s]",
            ""
        )
    )

    sample_nlp = tokenizer.transform(sample_nlp)

    sample_nlp = remover.transform(sample_nlp)

    # Aggregation

    sample_monthly = sample_nlp.groupBy(
        "year",
        "month"
    ).count()

    # Force execution
    sample_monthly.count()

    end = time.time()

    runtime = end - start

    print(f"Runtime: {runtime:.2f} seconds")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


 Benchmark: 10% Data 
72
Runtime: 3.60 seconds

 Benchmark: 50% Data 
72
Runtime: 2.48 seconds

 Benchmark: 100% Data 
72
Runtime: 2.35 seconds